# Phase 3 — SFT arm, no-trace variant (the first real training run)

**Runtime: L4.** ~1 h total, ~5 units. This is the tracer bullet: 1 seed,
end-to-end, so we see the whole teaching loop work before spending on 3 seeds.

**What it does:**
1. Builds the SFT mixture per Amendment A2: all train bugs, with the 97 never-solved
   (sft-pile) bugs upweighted 3x, plus ~17% verified-clean restraint examples
   (correct answer = return the function unchanged)
2. QLoRA-trains Qwen2.5-Coder-1.5B-Instruct (r=32, alpha=64, loss on the answer only)
3. Evaluates on the DEV slice (61 held-in bugs) at three points — before training,
   after epoch 1, after epoch 2 — reporting pass@1, pass@16, and the GAP
   (the amplification headroom RL needs; A2: never let SFT collapse it)

Targets are the factory's certified gold fixes — no teacher model involved.

In [ ]:
# Setup: Drive, fresh repo, data, routing
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, random, importlib
PHASE2 = '/content/drive/MyDrive/rl-post-training/phase2'
PHASE3 = '/content/drive/MyDrive/rl-post-training/phase3'
os.makedirs(PHASE3, exist_ok=True)
!rm -rf /content/ptl
!git clone -q https://github.com/nidhi1603/post-training-lab.git /content/ptl
sys.path.insert(0, '/content/ptl/src')
for mod in ('prompts',):
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])
from prompts import build_repair_prompt, extract_code
ver = !git -C /content/ptl log --oneline -1
print('FACTORY VERSION:', ver[0])
bugs = json.load(open('/content/ptl/data/data_v0_bugs.json'))
restraint = json.load(open('/content/ptl/data/data_v0_restraint.json'))
routing = {r['id']: r['pile'] for r in json.load(open(f'{PHASE2}/routing_v0.json'))}
train_bugs = [b for b in bugs if b['split'] == 'train']
dev_bugs = [b for b in bugs if b['split'] == 'dev']
print(len(train_bugs), 'train bugs |', len(dev_bugs), 'dev bugs |', len(routing), 'routed')

In [ ]:
# Build the SFT mixture (A2): sft-pile x3, others x1, + ~17% restraint
SEED = 3407
rng = random.Random(SEED)

def bug_example(b):
    return {'prompt': build_repair_prompt(b['buggy'], b['test_list']),
            'answer': '```python\n' + b['fixed'].strip() + '\n```'}

mixture = []
for b in train_bugs:
    copies = 3 if routing.get(b['id']) == 'sft' else 1
    mixture += [bug_example(b)] * copies
clean_train = [r for r in restraint if r['split'] == 'train']
rng.shuffle(clean_train)
for r in clean_train[:150]:
    mixture.append({'prompt': build_repair_prompt(r['code'], r['test_list']),
                    'answer': '```python\n' + r['code'].strip() + '\n```'})
rng.shuffle(mixture)
n_restraint = 150
print(f'{len(mixture)} examples ({n_restraint} restraint = {n_restraint/len(mixture)*100:.0f}%)')

In [ ]:
%%capture
!pip install unsloth

In [ ]:
# Load model 4-bit + LoRA (r=32, alpha=64 — the plan's config)
from unsloth import FastLanguageModel
import torch
model, tok = FastLanguageModel.from_pretrained(
    'unsloth/Qwen2.5-Coder-1.5B-Instruct', max_seq_length=1024,
    load_in_4bit=True, dtype=None)
model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=64, lora_dropout=0,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth', random_state=SEED)
print('LoRA attached (untrained LoRA == base model, so pre-training eval = baseline)')

In [ ]:
# Dataset in chat format + loss masked to the ANSWER only
from datasets import Dataset

def to_text(ex):
    msgs = [{'role': 'user', 'content': ex['prompt']},
            {'role': 'assistant', 'content': ex['answer']}]
    return {'text': tok.apply_chat_template(msgs, tokenize=False)}

ds = Dataset.from_list(mixture).map(to_text)
print('ONE FORMATTED EXAMPLE (eyeball it — user prompt, then assistant fix):')
print(ds[0]['text'][:1200])

In [ ]:
# Dev-slice evaluation: k=16 attempts at temp 1.0, graded by execution
import subprocess, tempfile
from concurrent.futures import ThreadPoolExecutor

def passes(code, bug, timeout=6):
    prog = '\n'.join(list(bug['test_imports']) + [code] + list(bug['test_list']))
    with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False) as f:
        f.write(prog); path = f.name
    try:
        return subprocess.run([sys.executable, path], capture_output=True, timeout=timeout).returncode == 0
    except subprocess.TimeoutExpired:
        return False
    finally:
        os.unlink(path)

@torch.no_grad()
def dev_eval(tag, k=16, batch_prompts=2):
    FastLanguageModel.for_inference(model)
    per_bug = []
    for i in range(0, len(dev_bugs), batch_prompts):
        chunk = dev_bugs[i:i+batch_prompts]
        texts = [tok.apply_chat_template([{'role':'user','content':build_repair_prompt(b['buggy'], b['test_list'])}],
                 tokenize=False, add_generation_prompt=True) for b in chunk]
        enc = tok(texts, return_tensors='pt', padding=True, padding_side='left').to('cuda')
        out = model.generate(**enc, do_sample=True, temperature=1.0, top_p=0.95,
                             num_return_sequences=k, max_new_tokens=512,
                             pad_token_id=tok.eos_token_id)
        gen = tok.batch_decode(out[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
        for j, b in enumerate(chunk):
            codes = [extract_code(t) for t in gen[j*k:(j+1)*k]]
            with ThreadPoolExecutor(max_workers=4) as pool:
                flags = list(pool.map(lambda c: passes(c, b), codes))
            per_bug.append({'id': b['id'], 'n': k, 'c': sum(flags)})
    p1 = sum(r['c']/r['n'] for r in per_bug) / len(per_bug)
    p16 = sum(1 for r in per_bug if r['c'] > 0) / len(per_bug)
    res = {'tag': tag, 'pass@1': p1, 'pass@16': p16, 'gap': p16 - p1, 'per_bug': per_bug}
    json.dump(res, open(f'{PHASE3}/dev_eval_{tag}.json', 'w'))
    print(f"[{tag}]  pass@1={p1*100:.1f}%  pass@16={p16*100:.1f}%  headroom gap={(p16-p1)*100:.1f}")
    return res

base_res = dev_eval('base_seed%d' % SEED)

In [ ]:
# TRAIN epoch 1 -> eval -> TRAIN epoch 2 -> eval  (loss on answers only)
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

def make_trainer():
    FastLanguageModel.for_training(model)
    t = SFTTrainer(model=model, tokenizer=tok, train_dataset=ds,
        dataset_text_field='text',
        args=SFTConfig(per_device_train_batch_size=8, gradient_accumulation_steps=2,
            num_train_epochs=1, learning_rate=2e-4, lr_scheduler_type='cosine',
            warmup_ratio=0.05, logging_steps=10, seed=SEED, output_dir='/content/out',
            report_to='none', save_strategy='no'))
    return train_on_responses_only(t,
        instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')

make_trainer().train()
model.save_pretrained(f'{PHASE3}/sft_notrace_s{SEED}_ep1')
ep1_res = dev_eval('ep1_seed%d' % SEED)

make_trainer().train()
model.save_pretrained(f'{PHASE3}/sft_notrace_s{SEED}_ep2')
ep2_res = dev_eval('ep2_seed%d' % SEED)

In [ ]:
# VERDICT: the Phase-3 gate + RL-init selection logic
print('tag        pass@1   pass@16   gap')
for r in (base_res, ep1_res, ep2_res):
    print(f"{r['tag']:<10} {r['pass@1']*100:6.1f}%  {r['pass@16']*100:6.1f}%  {r['gap']*100:5.1f}")
print('\nGate: SFT must beat base on dev pass@1.')
print('RL-init choice: highest pass@16 (the reachable set), NOT highest pass@1.')
print('Watch: if ep2 gap << ep1 gap, epoch 2 is eating the headroom RL needs (A2).')
print('\nAdapters saved to Drive phase3/. Bring all three rows back to the session.')